In [4]:
import duckdb

con = duckdb.connect("../data/market_data.duckdb", read_only=True)

# %% Active trades: breakout triggered in current trend session, still holding
df_active = con.execute("""
    SELECT
        ticker, company_name, sector, industry,
        ROUND(market_cap / 1e9, 2) AS mcap_bn,
        entry_date, ROUND(entry_price, 2) AS entry_price,
        ROUND(close_price, 2) AS current_close,
        ROUND(pct_return, 2) AS pct_return,
        days_held
    FROM v_screener_dashboard
    WHERE status = 'ACTIVE'
    ORDER BY entry_date, ticker
""").fetchdf()
print(f"Active trades: {len(df_active)}")
df_active

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Active trades: 186


,ticker,company_name,sector,industry,mcap_bn,entry_date,entry_price,current_close,pct_return,days_held
0,ATI,ATI Inc.,Industrials,Manufacturing - Metal Fabrication,19.42,2025-10-23,89.55,143.94,60.74,154
1,DSGN,"Design Therapeutics, Inc.",Healthcare,Biotechnology,0.59,2025-11-07,7.56,10.71,41.67,139
2,OUT,Outfront Media Inc.,Real Estate,REIT - Specialty,4.63,2025-11-13,21.70,26.75,23.27,133
3,TYRA,"Tyra Biosciences, Inc.",Healthcare,Biotechnology,2.00,2025-11-20,21.00,38.03,81.10,126
4,ROST,"Ross Stores, Inc.",Consumer Cyclical,Apparel - Retail,68.68,2025-11-21,174.00,214.30,23.16,125
...,...,...,...,...,...,...,...,...,...,...
181,CARE,"Carter Bankshares, Inc.",Financial Services,Banks - Regional,0.44,2026-03-26,21.94,21.94,0.00,0
182,CRUS,"Cirrus Logic, Inc.",Technology,Semiconductors,6.94,2026-03-26,148.69,148.69,0.00,0
183,KALV,"KalVista Pharmaceuticals, Inc.",Healthcare,Biotechnology,0.86,2026-03-26,18.95,18.95,0.00,0
184,KOD,Kodiak Sciences Inc.,Healthcare,Biotechnology,1.18,2026-03-26,39.76,39.76,0.00,0


In [5]:
# %% Watchlist: tickers in SEPA trend today but no breakout yet in this session
df_watchlist = con.execute("""
    WITH trend_sessions AS (
        SELECT
            t2.ticker, t2.date, t2.trend_ok, t2.breakout_ok,
            CASE WHEN t2.trend_ok AND NOT COALESCE(
                LAG(t2.trend_ok) OVER (PARTITION BY t2.ticker ORDER BY t2.date),
                FALSE
            ) THEN 1 ELSE 0 END AS session_start
        FROM t2_screener_features t2
        INNER JOIN company_profiles c ON t2.ticker = c.ticker
        WHERE c.is_active = TRUE
    ),
    sessions AS (
        SELECT ticker, date, breakout_ok,
            SUM(session_start) OVER (PARTITION BY ticker ORDER BY date) AS session_id
        FROM trend_sessions WHERE trend_ok
    ),
    current_sessions AS (
        SELECT ticker, session_id
        FROM sessions
        WHERE date = (SELECT MAX(date) FROM t2_screener_features)
    ),
    session_stats AS (
        SELECT s.ticker, s.session_id,
               MIN(s.date) AS session_start_date,
               BOOL_OR(s.breakout_ok) AS had_breakout
        FROM sessions s
        INNER JOIN current_sessions cs
            ON s.ticker = cs.ticker AND s.session_id = cs.session_id
        GROUP BY s.ticker, s.session_id
    )
    SELECT
        ss.ticker,
        cp.name AS company_name,
        cp.sector,
        cp.industry,
        ROUND(cp.market_cap / 1e9, 2) AS mcap_bn,
        ss.session_start_date,
        CAST(datediff('day', ss.session_start_date, p.date) AS INTEGER) AS days_in_trend,
        ROUND(p.close, 2) AS current_close
    FROM session_stats ss
    INNER JOIN company_profiles cp ON ss.ticker = cp.ticker
    INNER JOIN price_data p ON ss.ticker = p.ticker
        AND p.date = (SELECT MAX(date) FROM price_data WHERE ticker = ss.ticker)
    WHERE NOT ss.had_breakout
    ORDER BY ss.session_start_date, ss.ticker
""").fetchdf()
print(f"Watchlist (trend, no breakout yet): {len(df_watchlist)}")
df_watchlist


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Watchlist (trend, no breakout yet): 154


,ticker,company_name,sector,industry,mcap_bn,session_start_date,days_in_trend,current_close
0,STRL,"Sterling Infrastructure, Inc.",Industrials,Engineering & Construction,12.34,2026-01-16,69,415.93
1,NFBK,"Northfield Bancorp, Inc.",Financial Services,Banks - Regional,0.55,2026-02-20,34,13.40
2,INVA,"Innoviva, Inc.",Healthcare,Biotechnology,1.40,2026-02-25,29,22.66
3,PGC,Peapack-Gladstone Financial Corporation,Financial Services,Banks - Regional,0.58,2026-02-25,29,34.83
4,AL,Air Lease Corporation,Industrials,Rental & Leasing Services,7.24,2026-02-26,28,64.72
...,...,...,...,...,...,...,...,...
149,MAC,The Macerich Company,Real Estate,REIT - Retail,4.65,2026-03-26,0,19.09
150,MRVL,"Marvell Technology, Inc.",Technology,Semiconductors,76.86,2026-03-26,0,97.68
151,ORIC,"ORIC Pharmaceuticals, Inc.",Healthcare,Biotechnology,1.12,2026-03-26,0,12.38
152,UFCS,"United Fire Group, Inc.",Financial Services,Insurance - Property & Casualty,0.93,2026-03-26,0,37.24


In [6]:
# %% Recent exits (last 30 days)
df_exits = con.execute("""
    SELECT
        ticker, company_name, sector, industry,
        ROUND(market_cap / 1e9, 2) AS mcap_bn,
        entry_date, exit_date,
        ROUND(entry_price, 2) AS entry_price,
        ROUND(close_price, 2) AS exit_price,
        ROUND(pct_return, 2) AS pct_return,
        days_held
    FROM v_screener_dashboard
    WHERE status = 'EXITED'
      AND exit_date >= CURRENT_DATE - INTERVAL '30 days'
    ORDER BY exit_date DESC, pct_return DESC
""").fetchdf()
print(f"Recent exits (last 30 days): {len(df_exits)}")
df_exits

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Recent exits (last 30 days): 440


,ticker,company_name,sector,industry,mcap_bn,entry_date,exit_date,entry_price,exit_price,pct_return,days_held
0,FDX,FedEx Corporation,Industrials,Integrated Freight & Logistics,84.67,2025-12-18,2026-03-25,287.12,357.52,24.52,97
1,GHM,Graham Corporation,Industrials,Industrial - Machinery,0.89,2025-12-23,2026-03-25,69.50,85.62,23.19,92
2,AGX,"Argan, Inc.",Industrials,Engineering & Construction,6.58,2026-01-16,2026-03-25,383.66,437.48,14.03,68
3,ISSC,"Innovative Aerosystems, Inc.",Industrials,Aerospace & Defense,0.51,2026-02-18,2026-03-25,23.52,26.03,10.67,35
4,COKE,"Coca-Cola Consolidated, Inc.",Consumer Defensive,Beverages - Non-Alcoholic,16.86,2026-02-19,2026-03-25,176.79,186.34,5.40,34
...,...,...,...,...,...,...,...,...,...,...,...
435,LFST,"LifeStance Health Group, Inc.",Healthcare,Medical - Care Facilities,2.49,2026-02-25,2026-02-25,7.41,7.41,0.00,0
436,NVDA,NVIDIA Corporation,Technology,Semiconductors,4197.47,2026-02-25,2026-02-25,195.55,195.55,0.00,0
437,UHS,"Universal Health Services, Inc.",Healthcare,Medical - Care Facilities,11.81,2026-02-11,2026-02-25,231.32,230.51,-0.35,14
438,HEI,HEICO Corporation,Industrials,Aerospace & Defense,38.30,2026-02-20,2026-02-25,351.66,344.72,-1.97,5


In [7]:
# %%
con.close()